# F1 Skill Separation via Bayesian PGM — Report Notebook

**Course:** DTU MBML (02466)  
**Data:** 2011–2024 F1 seasons (286 races, 77 drivers, 17 constructors)  
**Models:** Three-tier Plackett-Luce PGM with sum-to-zero constructor constraints  
**Inference:** SVI (mean-field guide) + NUTS validation on Model 1  
**Generated:** 2026-05-11  
**Pipeline runtime:** ~15 min on M1 Pro CPU

This notebook contains the **complete pipeline** — model definitions, inference code, posterior results, validation diagnostics, and cross-model comparisons. It serves as the single source of truth for the report.


---

## 1. Setup and Data Overview


In [ ]:
import os, sys, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, MCMC, NUTS, Trace_ELBO
from IPython.display import Image, display

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)

OUTPUT_DIR = 'outputs/pgm_model'
PLOTS_DIR = os.path.join(OUTPUT_DIR, 'plots')
REPORT_DIR = os.path.join(PLOTS_DIR, 'report')

print(f'Pyro version: {pyro.__version__}')
print(f'PyTorch version: {torch.__version__}')


---

## 0. Pipeline Caching

The full pipeline (train all 3 models via SVI + NUTS) takes ~15 minutes. If posterior CSVs already exist on disk, we skip training and load from cache. To force retraining, delete `outputs/pgm_model/*.csv` and rerun this cell.


In [ ]:
import subprocess

REQUIRED_CSVS = [
    'baseline_posterior.csv',
    'extended_posterior.csv',
    'full_posterior.csv',
    'nuts_vs_svi_comparison.csv',
]

all_exist = all(os.path.exists(os.path.join('outputs/pgm_model', f)) for f in REQUIRED_CSVS)

if all_exist:
    print(f'[CACHE HIT] All {len(REQUIRED_CSVS)} posterior CSVs found in outputs/pgm_model/')
    print('           Skipping training. Delete CSVs to force retraining.')
else:
    missing = [f for f in REQUIRED_CSVS if not os.path.exists(os.path.join('outputs/pgm_model', f))]
    print(f'[CACHE MISS] Missing: {missing}')
    print('Running full pipeline...')
    t0 = time.time()
    result = subprocess.run(
        ['uv', 'run', 'python', '-m', 'models.pgm_backend.run_pgm'],
        capture_output=False, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f'Pipeline failed with exit code {result.returncode}')
    print(f'Pipeline complete ({time.time() - t0:.0f}s).')


---

## 2. Data Pipeline

### 2.1 Constructor Rebranding

The Ergast database assigns new `constructorId` values at each rebranding. We merge them into canonical IDs to preserve AR(1) continuity:


In [ ]:
CONSTRUCTOR_REMAP = {
    211: 10,   # Racing Point -> Force India
    117: 10,   # Aston Martin -> Force India
    214: 4,    # Alpine -> Renault
    213: 5,    # AlphaTauri -> Toro Rosso
    215: 5,    # Racing Bulls -> Toro Rosso
    51:  15,   # Alfa Romeo -> Sauber
}
print('6 rebranding merges applied.')


### 2.2 DNF Classification

Each entry is classified as Finished, Mechanical DNF, or Driver-Fault DNF. Mechanical DNFs are excluded from the Plackett-Luce ranking in Models 1 & 2 (asymmetric bias — see report for details). Model 3 includes them via a separate Bernoulli reliability term.


In [ ]:
MECHANICAL_STATUS_IDS = frozenset({
    5, 6, 7, 8, 9, 10, 21, 22, 26, 28, 29, 31, 36,
    40, 41, 43, 44, 54, 61, 65, 66, 67, 72, 75, 82, 104, 107, 108, 131,
})
print(f'{len(MECHANICAL_STATUS_IDS)} mechanical status IDs')


### 2.3 Pit-Stop Normalisation

Per-season z-scoring with zero-imputation (drivers with no pit stops get 0) and 99th-percentile winsorisation to handle extreme data artefacts.


In [ ]:
from models.pgm_backend.data_preparation import load_dataset
dataset = load_dataset()

mech_rate = dataset.is_mech.float().mean().item()
print(f'D={dataset.n_drivers}, K={dataset.n_constructors}, T={dataset.n_seasons}, C={dataset.n_circuits}, R={dataset.n_races}')
print(f'N_entries (ranking): {dataset.driver_idx.shape[0]}')
print(f'Mechanical DNF rate: {mech_rate:.4f} ({mech_rate*100:.1f}%)')
print(f'Wet races: {dataset.wet.sum().item():.0f} / {dataset.n_races}')


---

## 3. Plackett-Luce Likelihood

All three models share the same ranking likelihood. For a race with $N$ drivers, the probability of the observed finishing order $\pi$ is:

$$\log P(\pi \mid p) = \sum_{i=1}^{N} \left[p_{\pi(i)} - \log\sum_{j=i}^{N} \exp(p_{\pi(j)})\right]$$

Implemented via `torch.logsumexp` for numerical stability. This is the exact likelihood for the observed ordering — not an approximation.


In [ ]:
from models.pgm_backend.likelihood import plackett_luce_log_prob

p_test = torch.tensor([2.0, 1.0, 0.0])
r_test = torch.tensor([3])
ll = plackett_luce_log_prob(p_test, r_test)
print(f'3-driver hand-check log-prob: {ll:.4f}')  # approx -0.7209


---

## 4. Model 1 — Baseline (Static Skills)

### 4.1 Generative Model

The simplest instantiation of the skill-separation problem. One scalar skill per driver and one scalar performance per constructor, both fixed across all 14 seasons.

**Latent variables:** $s_d \sim \mathcal{N}(0, 1)$ for $D$ drivers, $c_k$ for $K$ constructors (sum-to-zero via reparameterisation)

**Performance:** $p_{d,r} = s_d + c_{k(d,r)}$

**Scientific question:** Can we separate driver from car at all?


In [ ]:
SIGMA_S = 1.0
SIGMA_C = 1.0

class BaselineModel:
    def __init__(self, n_drivers, n_constructors):
        self.D = n_drivers
        self.K = n_constructors

    def model(self, driver_idx, cons_idx, race_lengths):
        D, K = self.D, self.K
        s = pyro.sample('s', dist.Normal(0.0, SIGMA_S).expand([D]).to_event(1))
        c_raw = pyro.sample('c_raw', dist.Normal(0.0, SIGMA_C).expand([K - 1]).to_event(1))
        c = torch.cat([c_raw, -c_raw.sum(dim=0, keepdim=True)])  # sum-to-zero
        p = s[driver_idx] + c[cons_idx]
        log_prob = plackett_luce_log_prob(p, race_lengths)
        pyro.factor('race_obs', log_prob)

    def guide(self, driver_idx, cons_idx, race_lengths):
        D, K = self.D, self.K
        s_loc = pyro.param('s_loc', torch.zeros(D))
        s_scale = pyro.param('s_scale', torch.ones(D), constraint=dist.constraints.positive)
        pyro.sample('s', dist.Normal(s_loc, s_scale).to_event(1))
        c_loc = pyro.param('c_loc', torch.zeros(K - 1))
        c_scale = pyro.param('c_scale', torch.ones(K - 1), constraint=dist.constraints.positive)
        pyro.sample('c_raw', dist.Normal(c_loc, c_scale).to_event(1))

print('BaselineModel defined.')


### 4.2 SVI Training

3000 steps, ClippedAdam (lr=0.01, clip_norm=10), mean-field guide, Trace_ELBO.


In [ ]:
def train_svi(model, dataset, n_steps=3000, lr=0.01, step_kwargs=None):
    optimizer = pyro.optim.ClippedAdam({'lr': lr, 'clip_norm': 10.0})
    svi = SVI(model.model, model.guide, optimizer, loss=Trace_ELBO(num_particles=1))
    if step_kwargs is None:
        step_kwargs = {'driver_idx': dataset.driver_idx, 'cons_idx': dataset.cons_idx,
                       'race_lengths': dataset.race_lengths}
    losses = []
    for step in range(n_steps):
        loss = svi.step(**step_kwargs)
        losses.append(loss)
        if step % 500 == 0:
            print(f'  Step {step:5d}  ELBO: {loss:.2f}')
    return losses

print('train_svi() defined.')


### 4.3 Static Driver Rankings (from saved posterior)

In [ ]:
DRIVER_NAMES = {
    1: 'Hamilton', 830: 'Verstappen', 4: 'Alonso', 20: 'Vettel', 3: 'Rosberg',
    822: 'Leclerc', 832: 'Sainz', 846: 'Norris', 857: 'Piastri', 847: 'Russell',
    815: 'Ricciardo', 8: 'Raikkonen', 13: 'Webber', 18: 'Massa', 5: 'Button',
    811: 'Perez', 808: 'Grosjean', 807: 'Bottas', 24: 'Hulkenberg',
}

df_base = pd.read_csv(os.path.join(OUTPUT_DIR, 'baseline_posterior.csv'))
drivers_base = df_base[df_base.entity_type == 'driver'].sort_values('mu', ascending=False).reset_index(drop=True)
drivers_base.index = drivers_base.index + 1
drivers_base['name'] = drivers_base['entity_id'].apply(lambda x: DRIVER_NAMES.get(x, f'#{x}'))

display(drivers_base[['name', 'entity_id', 'mu', 'sigma']].head(20)
        .rename(columns={'mu': 's_mean', 'sigma': 's_std'})
        .style.set_caption('Model 1: Top 20 Static Driver Skills')
        .format({'s_mean': '{:+.4f}', 's_std': '{:.4f}'))


### 4.4 Static Constructor Rankings

In [ ]:
CONSTRUCTOR_NAMES = {
    131: 'Mercedes', 9: 'Red Bull', 6: 'Ferrari', 1: 'McLaren',
    3: 'Williams', 4: 'Renault', 5: 'Toro Rosso', 10: 'Force India',
    15: 'Sauber', 206: 'Haas', 164: 'Caterham', 166: 'Marussia', 205: 'Manor',
}

cons_base = df_base[df_base.entity_type == 'constructor'].sort_values('mu', ascending=False)
cons_base['name'] = cons_base['entity_id'].apply(lambda x: CONSTRUCTOR_NAMES.get(x, f'#{x}'))

display(cons_base[['name', 'entity_id', 'mu', 'sigma']]
        .rename(columns={'mu': 'c_mean', 'sigma': 'c_std'})
        .style.set_caption('Model 1: Constructor Performance Rankings')
        .format({'c_mean': '{:+.4f}', 'c_std': '{:.4f}'))

c_sum = cons_base['mu'].sum()
print(f'Sum-to-zero check: Σ c_k = {c_sum:.6f}')


---

## 5. Model 2 — Extended (Temporal Only)

### 5.1 Generative Model

Replaces static skills with season-level AR(1) random walks. The scientific question is purely: **do skills change over time?** No covariates, no additional latent structure.

**Latent variables:**
- $s_{d,t}$ — driver skill per season, AR(1): $s_{d,0} \sim \mathcal{N}(0, \sigma_s)$, $s_{d,t} \sim \mathcal{N}(s_{d,t-1}, \gamma_s)$
- $c_{k,t}$ — constructor performance per season, AR(1) with sum-to-zero per season

**Performance:** $p_{d,r} = s_{d, t(r)} + c_{k(d,r), t(r)}$

**Implementation:** AR(1) via cumsum of innovation vectors — no recursive `pyro.sample` loop. Innovation variance: $\gamma_s = 0.3$ (drivers), $\gamma_c = 0.5$ (constructors).


In [ ]:
GAMMA_S = 0.3
GAMMA_C = 0.5

class ExtendedModel:
    def __init__(self, n_drivers, n_constructors, n_seasons):
        self.D = n_drivers
        self.K = n_constructors
        self.T = n_seasons

    def model(self, driver_idx, cons_idx, season_idx, race_lengths):
        D, K, T = self.D, self.K, self.T
        
        s0 = pyro.sample('s0', dist.Normal(0.0, SIGMA_S).expand([D]).to_event(1))
        s_innov = pyro.sample('s_innov', dist.Normal(0.0, GAMMA_S).expand([T-1, D]).to_event(2))
        s = torch.cat([s0.unsqueeze(0), s0.unsqueeze(0) + s_innov.cumsum(0)], dim=0)  # (T,D)
        
        c0_raw = pyro.sample('c0_raw', dist.Normal(0.0, SIGMA_C).expand([K-1]).to_event(1))
        c_innov = pyro.sample('c_innov', dist.Normal(0.0, GAMMA_C).expand([T-1, K-1]).to_event(2))
        c_raw = torch.cat([c0_raw.unsqueeze(0), c0_raw.unsqueeze(0) + c_innov.cumsum(0)], dim=0)
        c = torch.cat([c_raw, -c_raw.sum(dim=1, keepdim=True)], dim=1)  # (T,K), sum-to-zero
        
        p = s[season_idx, driver_idx] + c[season_idx, cons_idx]
        log_prob = plackett_luce_log_prob(p, race_lengths)
        pyro.factor('race_obs', log_prob)

    def guide(self, driver_idx, cons_idx, season_idx, race_lengths):
        D, K, T = self.D, self.K, self.T
        s0_loc = pyro.param('s0_loc', torch.zeros(D))
        s0_scale = pyro.param('s0_scale', torch.ones(D), constraint=dist.constraints.positive)
        pyro.sample('s0', dist.Normal(s0_loc, s0_scale).to_event(1))
        s_innov_loc = pyro.param('s_innov_loc', torch.zeros(T-1, D))
        s_innov_scale = pyro.param('s_innov_scale', torch.ones(T-1, D), constraint=dist.constraints.positive)
        pyro.sample('s_innov', dist.Normal(s_innov_loc, s_innov_scale).to_event(2))
        c0_raw_loc = pyro.param('c0_raw_loc', torch.zeros(K-1))
        c0_raw_scale = pyro.param('c0_raw_scale', torch.ones(K-1), constraint=dist.constraints.positive)
        pyro.sample('c0_raw', dist.Normal(c0_raw_loc, c0_raw_scale).to_event(1))
        c_innov_loc = pyro.param('c_innov_loc', torch.zeros(T-1, K-1))
        c_innov_scale = pyro.param('c_innov_scale', torch.ones(T-1, K-1), constraint=dist.constraints.positive)
        pyro.sample('c_innov', dist.Normal(c_innov_loc, c_innov_scale).to_event(2))

print('ExtendedModel defined (temporal only, no circuit/weather).')


### 5.2 ELBO Convergence

In [ ]:
display(Image(os.path.join(PLOTS_DIR, 'elbo_curves.png')))


### 5.3 Constructor Performance Trajectories

The temporal model recovers three regulation-era transitions without external labels:


In [ ]:
display(Image(os.path.join(REPORT_DIR, 'fig2_constructor_trajectories.png')))


### 5.4 Temporal Constructor Rankings by Season

In [ ]:
df_ext = pd.read_csv(os.path.join(OUTPUT_DIR, 'extended_posterior.csv'))
seasons_to_show = [0, 3, 8, 13]  # 2011, 2014, 2019, 2024
season_labels = {0: '2011', 3: '2014', 8: '2019', 13: '2024'}

for s in seasons_to_show:
    sdata = df_ext[(df_ext.entity_type == 'constructor') & (df_ext.season == s)]
    sdata = sdata.sort_values('mu', ascending=False).reset_index(drop=True)
    sdata.index = sdata.index + 1
    sdata['name'] = sdata['entity_id'].apply(lambda x: CONSTRUCTOR_NAMES.get(x, f'#{x}'))
    display(sdata[['name', 'mu', 'sigma']].head(8)
            .rename(columns={'mu': 'c_mean', 'sigma': 'c_std'})
            .style.set_caption(f'Model 2 Constructor Rankings — {season_labels[s]}')
            .format({'c_mean': '{:+.4f}', 'c_std': '{:.4f}'))
    print(f'  Σ c_k ({season_labels[s]}): {sdata["mu"].sum():.6f}')


### 5.5 Temporal Driver Rankings — 2024

In [ ]:
m2_2024 = df_ext[(df_ext.entity_type == 'driver') & (df_ext.season == 13)]
m2_2024 = m2_2024.sort_values('mu', ascending=False).reset_index(drop=True)
m2_2024.index = m2_2024.index + 1
m2_2024['name'] = m2_2024['entity_id'].apply(lambda x: DRIVER_NAMES.get(x, f'#{x}'))
display(m2_2024[['name', 'entity_id', 'mu', 'sigma']].head(20)
        .rename(columns={'mu': 's_mean', 'sigma': 's_std'})
        .style.set_caption('Model 2: 2024 Temporal Driver Rankings')
        .format({'s_mean': '{:+.4f}', 's_std': '{:.4f}'))


---

## 6. Model 3 — Full

### 6.1 Generative Model

Extends the temporal model with five additional components:

1. **Circuit effects** $e_c \sim \mathcal{N}(0, \sigma_e)$ — track-specific biases
2. **Global wet-weather** $\beta_w \sim \mathcal{N}(0, 0.5)$ — average rain effect
3. **Driver wet-weather interaction** $\delta_d \cdot w_r$ — driver-specific rain skill (multiplicative)
4. **Pit-stop covariate** $\beta_\pi \cdot \pi_{d,r}$ — normalised pit-stop duration
5. **Bernoulli reliability** $\text{sigmoid}(-\alpha_\text{rel} - c_k)$ — mechanical DNF probability

**Performance:** $p_{d,r} = s_{d,t(r)} + c_{k(d,r),t(r)} + e_{\text{circ}(r)} + \beta_w \cdot w_r + \delta_d \cdot w_r + \beta_\pi \cdot \pi_{d,r}$

**Scientific question:** What additional structure exists in race outcomes beyond temporal skill dynamics?


In [ ]:
SIGMA_E = 0.5
SIGMA_DELTA = 0.5

class FullModel:
    def __init__(self, n_drivers, n_constructors, n_seasons, n_circuits):
        self.D = n_drivers; self.K = n_constructors
        self.T = n_seasons; self.C = n_circuits

    def model(self, driver_idx, cons_idx, season_idx, circuit_idx, race_idx,
              wet, race_lengths, pit_norm, is_mech, cons_idx_all, season_idx_all):
        D, K, T, C = self.D, self.K, self.T, self.C
        
        # AR(1) driver skills
        s0 = pyro.sample('s0', dist.Normal(0.0, SIGMA_S).expand([D]).to_event(1))
        s_innov = pyro.sample('s_innov', dist.Normal(0.0, GAMMA_S).expand([T-1, D]).to_event(2))
        s = torch.cat([s0.unsqueeze(0), s0.unsqueeze(0) + s_innov.cumsum(0)], dim=0)
        
        # AR(1) constructor skills with sum-to-zero
        c0_raw = pyro.sample('c0_raw', dist.Normal(0.0, SIGMA_C).expand([K-1]).to_event(1))
        c_innov = pyro.sample('c_innov', dist.Normal(0.0, GAMMA_C).expand([T-1, K-1]).to_event(2))
        c_raw = torch.cat([c0_raw.unsqueeze(0), c0_raw.unsqueeze(0) + c_innov.cumsum(0)], dim=0)
        c = torch.cat([c_raw, -c_raw.sum(dim=1, keepdim=True)], dim=1)
        
        # Circuit effects
        e_circ = pyro.sample('e_circ', dist.Normal(0.0, SIGMA_E).expand([C]).to_event(1))
        # Global weather
        beta_w = pyro.sample('beta_w', dist.Normal(0.0, 0.5))
        # Driver wet-weather modifier
        delta_d = pyro.sample('delta_d', dist.Normal(0.0, SIGMA_DELTA).expand([D]).to_event(1))
        # Pit-stop coefficient
        beta_pi = pyro.sample('beta_pi', dist.Normal(0.0, 0.5))
        # Reliability intercept
        alpha_rel = pyro.sample('alpha_rel', dist.Normal(0.0, 1.0))
        
        # Performance (ranking entries only)
        p = (s[season_idx, driver_idx] + c[season_idx, cons_idx]
             + e_circ[circuit_idx] + beta_w * wet[race_idx]
             + delta_d[driver_idx] * wet[race_idx] + beta_pi * pit_norm)
        log_prob = plackett_luce_log_prob(p, race_lengths)
        pyro.factor('race_obs', log_prob)
        
        # Mechanical DNF reliability (all rows)
        mech_prob = torch.sigmoid(-alpha_rel - c[season_idx_all, cons_idx_all])
        pyro.factor('reliability', dist.Bernoulli(mech_prob).log_prob(is_mech.float()).sum())

    def guide(self, driver_idx, cons_idx, season_idx, circuit_idx, race_idx,
              wet, race_lengths, pit_norm, is_mech, cons_idx_all, season_idx_all):
        D, K, T, C = self.D, self.K, self.T, self.C
        # ... (mean-field guide: independent Normals for each latent variable)
        s0_loc = pyro.param('s0_loc', torch.zeros(D))
        s0_scale = pyro.param('s0_scale', torch.ones(D), constraint=dist.constraints.positive)
        pyro.sample('s0', dist.Normal(s0_loc, s0_scale).to_event(1))
        s_innov_loc = pyro.param('s_innov_loc', torch.zeros(T-1, D))
        s_innov_scale = pyro.param('s_innov_scale', torch.ones(T-1, D), constraint=dist.constraints.positive)
        pyro.sample('s_innov', dist.Normal(s_innov_loc, s_innov_scale).to_event(2))
        c0_raw_loc = pyro.param('c0_raw_loc', torch.zeros(K-1))
        c0_raw_scale = pyro.param('c0_raw_scale', torch.ones(K-1), constraint=dist.constraints.positive)
        pyro.sample('c0_raw', dist.Normal(c0_raw_loc, c0_raw_scale).to_event(1))
        c_innov_loc = pyro.param('c_innov_loc', torch.zeros(T-1, K-1))
        c_innov_scale = pyro.param('c_innov_scale', torch.ones(T-1, K-1), constraint=dist.constraints.positive)
        pyro.sample('c_innov', dist.Normal(c_innov_loc, c_innov_scale).to_event(2))
        e_circ_loc = pyro.param('e_circ_loc', torch.zeros(C))
        e_circ_scale = pyro.param('e_circ_scale', torch.ones(C), constraint=dist.constraints.positive)
        pyro.sample('e_circ', dist.Normal(e_circ_loc, e_circ_scale).to_event(1))
        beta_w_loc = pyro.param('beta_w_loc', torch.tensor(0.0))
        beta_w_scale = pyro.param('beta_w_scale', torch.tensor(1.0), constraint=dist.constraints.positive)
        pyro.sample('beta_w', dist.Normal(beta_w_loc, beta_w_scale))
        delta_d_loc = pyro.param('delta_d_loc', torch.zeros(D))
        delta_d_scale = pyro.param('delta_d_scale', torch.ones(D), constraint=dist.constraints.positive)
        pyro.sample('delta_d', dist.Normal(delta_d_loc, delta_d_scale).to_event(1))
        beta_pi_loc = pyro.param('beta_pi_loc', torch.tensor(0.0))
        beta_pi_scale = pyro.param('beta_pi_scale', torch.tensor(1.0), constraint=dist.constraints.positive)
        pyro.sample('beta_pi', dist.Normal(beta_pi_loc, beta_pi_scale))
        alpha_rel_loc = pyro.param('alpha_rel_loc', torch.tensor(0.0))
        alpha_rel_scale = pyro.param('alpha_rel_scale', torch.tensor(1.0), constraint=dist.constraints.positive)
        pyro.sample('alpha_rel', dist.Normal(alpha_rel_loc, alpha_rel_scale))

print('FullModel defined.')


### 6.2 Model 3 Scalar Posteriors

In [ ]:
display(Image(os.path.join(REPORT_DIR, 'fig5_model3_scalars.png')))


In [ ]:
df_full = pd.read_csv(os.path.join(OUTPUT_DIR, 'full_posterior.csv'))

for name in ['beta_w', 'beta_pi', 'alpha_rel']:
    row = df_full[df_full.entity_name == name]
    if len(row):
        mu, sigma = row['mu'].values[0], row['sigma'].values[0]
        if name == 'alpha_rel':
            p_mech = float(torch.sigmoid(torch.tensor(-mu)))
            print(f'{name}: {mu:+.4f} ± {sigma:.4f}  →  sigmoid(−{mu:.2f}) = {p_mech:.3f} ({p_mech*100:.1f}%)')
        else:
            print(f'{name}: {mu:+.4f} ± {sigma:.4f}')


### 6.3 Wet-Weather Specialists ($\delta_d$)

In [ ]:
display(Image(os.path.join(PLOTS_DIR, 'wet_weather_specialists.png')))


In [ ]:
delta_data = df_full[df_full.entity_name == 'delta_d'].sort_values('mu', ascending=False).reset_index(drop=True)
delta_data.index = delta_data.index + 1
delta_data['name'] = delta_data['entity_id'].apply(lambda x: DRIVER_NAMES.get(x, f'#{x}'))
display(delta_data[['name', 'entity_id', 'mu', 'sigma']].head(15)
        .rename(columns={'mu': 'delta_mean', 'sigma': 'delta_std'})
        .style.set_caption('Model 3: Top 15 Wet-Weather Specialists')
        .format({'delta_mean': '{:+.4f}', 'delta_std': '{:.4f}'))

n_pos = (delta_data['mu'] > 0).sum()
print(f'{n_pos}/{len(delta_data)} drivers have positive delta_d')


### 6.4 Pit-Stop Coefficient Posterior

In [ ]:
display(Image(os.path.join(PLOTS_DIR, 'beta_pi_posterior.png')))


---

## 7. Inference Validation

### 7.1 NUTS on Model 1

NUTS (No-U-Turn Sampler) is run on Model 1 only (D + K − 1 ≈ 93 parameters) as an inference validation step. 500 warmup + 500 samples. If SVI and NUTS agree within reason, we trust SVI on Models 2 & 3.


In [ ]:
def run_nuts(model, dataset, num_warmup=500, num_samples=500):
    pyro.clear_param_store()
    kernel = NUTS(model.model)
    mcmc = MCMC(kernel, num_samples=num_samples, warmup_steps=num_warmup)
    mcmc.run(driver_idx=dataset.driver_idx, cons_idx=dataset.cons_idx,
             race_lengths=dataset.race_lengths)
    return mcmc

print('run_nuts() defined.')


### 7.2 SVI vs NUTS Scatter

In [ ]:
display(Image(os.path.join(PLOTS_DIR, 'svi_vs_nuts_scatter.png')))


In [ ]:
df_nuts = pd.read_csv(os.path.join(OUTPUT_DIR, 'nuts_vs_svi_comparison.csv'))
nuts_d = df_nuts[df_nuts.entity_type == 'driver']
nuts_c = df_nuts[df_nuts.entity_type == 'constructor']

rhat_vals = df_nuts['r_hat'].dropna().values
rhat_bad = (rhat_vals >= 1.05).sum()
print(f'R-hat: {rhat_bad}/{len(rhat_vals)} latents >= 1.05, max = {rhat_vals.max():.4f}')
print(f'Driver discrepancy: median={nuts_d.discrepancy.median():.3f}, max={nuts_d.discrepancy.max():.3f}')
print(f'Constructor discrepancy: median={nuts_c.discrepancy.median():.3f}, max={nuts_c.discrepancy.max():.3f}')


### 7.3 Synthetic Data Recovery

In [ ]:
display(Image(os.path.join(PLOTS_DIR, 'synthetic_recovery.png')))


### 7.4 Prior Predictive Check

In [ ]:
display(Image(os.path.join(PLOTS_DIR, 'prior_predictive_win_rate.png')))


### 7.5 Posterior Uncertainty vs Experience

In [ ]:
display(Image(os.path.join(PLOTS_DIR, 'uncertainty_vs_races.png')))


---

## 8. Cross-Model Comparisons

### 8.1 Static vs Temporal Driver Skills


In [ ]:
display(Image(os.path.join(REPORT_DIR, 'fig3_static_vs_temporal.png')))


### 8.2 Cross-Model Driver Ranking (Top 15, 2024)

In [ ]:
display(Image(os.path.join(PLOTS_DIR, 'cross_model_driver_ranking.png')))


---

## 9. Model Comparison Summary


In [ ]:
summary_data = {
    'Metric': ['Latent variables', 'Covariates', 'ELBO (final)',
               'Inference', 'Key finding'],
    'Model 1 (Baseline)': [
        's_d, c_k (static)', 'None', '~10,700',
        'SVI + NUTS', 'Mercedes dominates; Piastri #1 static'
    ],
    'Model 2 (Extended)': [
        's_{d,t}, c_{k,t} (AR(1))', 'None', '~11,200',
        'SVI only', 'Regulation-era transitions emerge from data'
    ],
    'Model 3 (Full)': [
        '+ e_circ, beta_w, delta_d, beta_pi, alpha_rel',
        'w_r (wet), pi_{d,r} (pit)', '~13,000',
        'SVI only', 'beta_w ≈ 0, beta_pi ≈ 0 (null); wet-skill varies'
    ]
}
pd.DataFrame(summary_data).style.set_caption('Three-Model Comparison').hide(axis='index')


---

## 10. Key Findings

### What the models do well
1. **Skill separation:** Plackett-Luce successfully separates driver skill from constructor performance. Sum-to-zero ensures identifiable, symmetric estimates.
2. **Temporal tracking:** AR(1) random walks capture regulation-driven regime changes (Mercedes hybrid era → Red Bull ground-effect era) without external labels.
3. **Consistency:** Top drivers (Verstappen, Hamilton, Alonso) and constructors (Mercedes, Red Bull, Ferrari) occupy expected positions.

### Surprising / null findings
1. **$\beta_w \approx -0.03 \pm 0.51$:** No global wet-weather effect — any rain signal is driver-specific.
2. **$\beta_\pi \approx 0.02 \pm 0.03$:** After correcting data artefacts, pit-stop duration has no detectable effect on race performance. The previously reported +0.26 was a data artefact.
3. **$\alpha_\text{rel} \approx 2.09$:** Baseline mechanical DNF probability ~11.0%. Better constructors fail mechanically less often.
4. **Wet-weather specialists differ from reputation:** Model's top $\delta_d$ drivers differ from historical 'rain masters' (Alonso, Webber). With only ~30 wet races, posterior uncertainty is high.
5. **Mean-field VI bias:** SVI underestimates posterior variance and shrinks constructor means more than driver means.

### Limitations
1. **No grid position:** Excluded as a blocking variable on the causal path. Cannot separate qualifying skill from race execution.
2. **Pit-stop confounding:** $\beta_\pi$ captures strategy + speed, not pure execution.
3. **Wet-weather data scarcity:** Only ~10% of races are wet. $\delta_d$ estimates have high posterior variance.
4. **Constructor reliability conflation:** In Model 3, $c_k$ captures both pace and reliability.
5. **Mean-field approximation:** A multivariate Normal guide would reduce SVI-NUTS discrepancy.
